# train.ipynb — 1 enc 3 dec 통합 학습 (담당: 세원)

1. 환경 준비 → 2. 설정 → 3. 데이터 → 4. 모델 → 5. 학습 → 6. 방향별 평가 → 7. 6개 파일 저장 → 8. 저장 파일 확인

`TODO(…)` 표시는 다른 파트 코드가 push되면 함수 이름·인자를 맞출 곳이다.

In [ ]:
# 1. 환경 준비 (Colab / 로컬 둘 다 동작)
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists("/content/2026-nlp"):
        !git -C /content/2026-nlp pull
    else:
        !git clone https://github.com/sloth2003/2026-nlp.git /content/2026-nlp
    REPO = "/content/2026-nlp"
    CKPT_DIR = "/content/drive/MyDrive/2026-nlp/checkpoints"   # 학습 중간 저장 (세션 끊김 대비)
else:
    REPO = os.path.abspath("../..")                              # notebook/ 기준
    CKPT_DIR = os.path.join(REPO, "project01-dev", "train")

DEV = os.path.join(REPO, "project01-dev")
SUBMIT_DIR = os.path.join(REPO, "project01")                    # 제출용 .pt 6개 위치
os.makedirs(CKPT_DIR, exist_ok=True)
sys.path.insert(0, os.path.join(DEV, "src"))
print(REPO, CKPT_DIR, SUBMIT_DIR, sep="\n")

In [ ]:
# 2. 설정
import json, random
import numpy as np
import torch

with open(os.path.join(DEV, "config", "split.json"), encoding="utf-8") as f:
    config = json.load(f)

SEED = config["random_seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| config:", config)

MODEL_KWARGS = dict(embedding_dim=128, hidden_dim=256, num_layers=2, dropout=0.1)   # model.py 인자와 같게

In [ ]:
# 3. 데이터
# TODO(재후 data.py): 함수 이름 맞추기. 필요한 반환 형식:
#   loaders   : loaders["kor2eng"]["train" / "valid" / "test"]  (6방향)
#   src_vocab : 세 언어 통합 입력 Vocab
#   tgt_vocabs: {"num": Vocab, "eng": Vocab, "kor": Vocab}
import data
from seq2seq import DIRECTIONS, LANGS

loaders, src_vocab, tgt_vocabs = data.load_all_data(config)      # TODO(재후)

print("입력 vocab:", len(src_vocab), "| 출력 vocab:", {l: len(v) for l, v in tgt_vocabs.items()})
for d in DIRECTIONS:
    print(f"{d:8s}", {k: len(loaders[d][k].dataset) for k in ("train", "valid", "test")})

In [ ]:
# 4. 모델: 공유 Encoder 1개 + Decoder 3개
from seq2seq import build_model

model = build_model(len(src_vocab), {l: len(v) for l, v in tgt_vocabs.items()}, **MODEL_KWARGS).to(device)
print(model)
print("파라미터:", f"{sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 5. 학습
# TODO(성현 train.py): 함수 이름·인자 맞추기. 안에서 할 일:
#   6방향 배치를 비율대로 섞어서 logits = model(x, y, tgt_lang), loss = batch_loss(y[:, 1:], logits[:, :-1])
#   epoch 마다 방향별 valid 평가, 끝날 때마다 save_training_state 로 중간 저장
from train import train
from checkpoint import save_training_state, load_training_state

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
LAST = os.path.join(CKPT_DIR, "last.pt")

start_epoch = 0
if os.path.exists(LAST):                                          # 세션이 끊겼다가 다시 실행한 경우 이어서
    start_epoch, best = load_training_state(LAST, model, optimizer, device)
    start_epoch += 1
    print("이어서 학습:", start_epoch, "epoch부터, best =", best)

history = train(model, loaders, optimizer, config, device=device,   # TODO(성현)
                start_epoch=start_epoch, ckpt_path=LAST)

In [ ]:
# 6. 방향별 test 평가 (beam)
from inference import evaluate

model.eval()
results = {}
for d in DIRECTIONS:
    _, tgt = d.split("2")
    r = evaluate(model.direction(d), loaders[d]["test"], tgt_vocabs[tgt],
                 method="beam", beam_width=5, src_vocab=src_vocab)
    results[d] = r["acc"]
    print(f"{d:8s} acc {r['acc']:.4f} ({r['correct']}/{r['total']})  틀린 예: {r['errors'][:2]}")

In [ ]:
# 7. 제출용 .pt 6개 저장 (project01/ + Drive 백업)
from checkpoint import save_all

paths = save_all(SUBMIT_DIR, model, src_vocab, tgt_vocabs, MODEL_KWARGS)
save_all(os.path.join(CKPT_DIR, "final"), model, src_vocab, tgt_vocabs, MODEL_KWARGS)
for p in paths:
    print(p, f"{os.path.getsize(p) / 1e6:.1f} MB")

In [ ]:
# 8. 저장한 파일만으로 다시 로드해서 번역 확인
from checkpoint import load_model
from inference import translate

examples = {"num": "12708", "kor": "만 이천칠백팔", "eng": "twelve thousand seven hundred eight"}
for d in DIRECTIONS:
    src, _ = d.split("2")
    m, sv, tv = load_model(os.path.join(SUBMIT_DIR, f"{d}.pt"), device=device)
    print(f"{d:8s} {examples[src]!r} -> {translate(m, examples[src], sv, tv, method='beam')!r}")